# Ovarian ultrasound classification: Colab / Kaggle runner

**Research and education only. Not a diagnostic tool.**

Runs the full pipeline on a free GPU (Runtime > Change runtime type > T4 GPU). All logic lives in the `follicle_ml` package; this notebook only calls the CLI. Data is downloaded into the runtime and never committed.

In [ ]:
!git clone https://github.com/armita-jamshidi/ML_follicular_tracking.git
%cd ML_follicular_tracking
!pip install -q -e '.[secondary]'

## 1. Download and audit
`audit` runs the perceptual-hash duplicate check, records which acquisition source each image comes from, and assigns duplicate-aware stratified folds. Training commands refuse to run without it.

In [ ]:
!follicle-ml download
!follicle-ml audit

## 2. Baselines
Shortcut model (image size and colour only), kNN on raw pixels, and kNN / logistic regression on frozen ImageNet embeddings.

In [ ]:
!follicle-ml baselines

## 3. Fine-tune the CNN (5-fold CV)
About a minute per fold on a T4. Switch backbone with `-s model.name=efficientnet_b0`.

In [ ]:
!follicle-ml train-cnn
!follicle-ml gradcam

## 4. Sensitivity analysis: standard scanner exports only
Most Normal images come from a different source than Dominant Follicle and PCO images, so a model can score well by recognising the source. Re-running on the `standard_export` subset removes that shortcut (at the cost of only 10 Normal images).

In [ ]:
!follicle-ml -s data.subset=standard_export baselines
!follicle-ml -s data.subset=standard_export train-cnn

## 5. Cross-dataset test (optional)
Trains on the whole main dataset and tests PCO vs. rest on the deduplicated BTX24 set. BTX24 is never used for training here.

In [ ]:
!follicle-ml download --secondary
!follicle-ml cross-dataset

## 6. Collect results

In [ ]:
!follicle-ml report
from IPython.display import Markdown, Image, display
display(Markdown(open('outputs/RESULTS.md').read()))
display(Image('outputs/cnn_resnet18/confusion_matrix.png'))
display(Image('outputs/cnn_resnet18/gradcam.png'))